# Résumé Energétique

## Config

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from omegaconf import OmegaConf
from pathlib import Path

load_dotenv(find_dotenv())
config_path = Path("configs/config.yaml")
cfg = OmegaConf.load(config_path)
print(OmegaConf.to_yaml(cfg, resolve=False))

gemini_model:
  name: gemini-2.5-flash
  api_key: ${oc.env:GEMINI_API_KEY}



## Récupération du Texte depuis le document PDF ou TXT

In [2]:
from pypdf import PdfReader

filename = "documents/RAPPORT.pdf" # Change name as needed - .pdf and .txt supported

text = ""
if(filename.endswith(".pdf")):
    try:
        reader = PdfReader(filename)
        for page in reader.pages:
            text += page.extract_text() or ""
    except Exception as e:
        print(f"An error occurred: {e}")
elif(filename.endswith(".txt")):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            text = file.read()
    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("Veuillez utiliser un fichier PDF ou TXT file. Le nom du fichier doit finir par '.pdf' ou '.txt'.")

## Résumé général: Compréhension globale du texte

In [4]:
from google import genai
client = genai.Client(api_key=cfg.gemini_model.api_key)
chat = client.chats.create(model=cfg.gemini_model.name)

In [5]:
summary_prompt_file = './prompts/resume_general.txt'
try:
    with open(summary_prompt_file, 'r', encoding='utf-8') as file:
        summary_prompt = file.read() + text
except FileNotFoundError:
    print(f"Erreur: Le fichier '{summary_prompt_file}' n'existe pas.")
except Exception as e:
    print(f"Une erreur s'est produite: {e}") 

In [6]:
summary = chat.send_message(summary_prompt)

In [7]:
import json
import re

summary_response_file = './results/summary.json'

def extract_json(text):
    match = re.search(r'(\{.*\}|\[.*\])', text, re.DOTALL)
    if not match:
        raise ValueError("Pas de JSON dans le texte.")

    json_str = match.group(1)
    json_str = re.sub(
        r"(?<=\{|,)\s*'([^']+)'\s*:",
        r'"\1":',
        json_str
    )
    json_str = re.sub(
        r":\s*'([^']*)'",
        r': "\1"',
        json_str
    )

    return json_str
try:
    summ = json.loads(extract_json(summary.text))
    with open(summary_response_file, 'w', encoding='utf-8') as file:
        json.dump(summ, file, indent=2)
except json.JSONDecodeError as e:
    print(f"Le JSON n'a pas pu etre décodé: {e}")
    print("Réessayons le prompt...")
    summary = chat.send_message(summary_prompt)
    summ = json.loads(extract_json(summary.text))
    with open(summary_response_file, 'w', encoding='utf-8') as file:
        json.dump(summ, file, indent=2)

## Extraction 

In [8]:
extraction_prompt_file = './prompts/extraction.txt'
summary_file = './results/summary.json'
try:
    with open(extraction_prompt_file, 'r', encoding='utf-8') as file:
        extraction_prompt = file.read()
except FileNotFoundError:
    print(f"Erreur: Le fichier '{extraction_prompt_file}' n'existe pas.")
except Exception as e:
    print(f"Une erreur s'est produite: {e}") 
    
try:
    with open(summary_file, 'r') as file:
        extraction_prompt += str(json.load(file))
except FileNotFoundError:
    print(f"Erreur: Le fichier '{summary_file}' n'existe pas.")
except Exception as e:
    print(f"Une erreur s'est produite: {e}") 

In [15]:
extracted_json = chat.send_message(extraction_prompt)

In [16]:
results_file = './results/result.json'

try:
    extr_json = json.loads(extract_json(extracted_json.text))
    with open(results_file, 'w', encoding='utf-8') as file:
        json.dump(extr_json, file, indent=2)
except json.JSONDecodeError as e:
    print(f"Le JSON n'a pas pu etre décodé: {e}")
    print("Réessayons le prompt...")
    extracted_json = chat.send_message(extraction_prompt)
    extr_json = json.loads(extract_json(extracted_json.text))
    with open(results_file, 'w', encoding='utf-8') as file:
        json.dump(extr_json, file, indent=2)

## Vérifications et tests

In [17]:
results_file = './results/result.json'

try:
    with open(results_file, 'r') as file:
        result = json.load(file)
except FileNotFoundError:
    print(f"Erreur: Le fichier '{results_file}' n'existe pas.")
except Exception as e:
    print(f"Une erreur s'est produite: {e}") 

In [18]:
from pypdf import PdfReader

filename = "documents/RAPPORT.pdf" # Change name as needed - .pdf and .txt supported

text = ""
if(filename.endswith(".pdf")):
    try:
        reader = PdfReader(filename)
        for page in reader.pages:
            text += page.extract_text() or ""
    except Exception as e:
        print(f"An error occurred: {e}")
elif(filename.endswith(".txt")):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            text = file.read()
    except FileNotFoundError:
        print(f"Erreur: Le fichier '{filename}' n'existe pas.")
    except Exception as e:
        print(f"Une erreur s'est produite: {e}") 
else:
    print("Veuillez utiliser un fichier PDF ou TXT file. Le nom du fichier doit finir par '.pdf' ou '.txt'.")

### Cohérence interne

In [20]:
if result["objectifs_climatiques"]["neutralite_carbone"]["engagement"] and not result["objectifs_climatiques"]["neutralite_carbone"]["annee_cible"]:
    print("Attention: année cible de neutralité manquante")

In [21]:
for reduction in result["objectifs_climatiques"]["reductions_emissions"]:
    if reduction["annee_reference"] and reduction["annee_cible"]:
        if reduction["annee_reference"] >= reduction["annee_cible"]:
            print(f"{reduction['entite']} {reduction['scope']}: L'année de référence doit etre avant l'année cible.")
    if reduction["valeur"] is not None:
        if reduction["valeur"] < 0 or reduction["valeur"] > 100:
            print(f"{reduction['entite']} {reduction['scope']}: Pourcentage de réduction hors plage raisonnable.")

In [22]:
for investissement in result["investissements_transition"]:
    if investissement["montant"] and not investissement["unite"]:
        print(f"{str(investissement)}: L'investissement a une valeur mais pas d'unité.")
    if investissement["montant"] and not investissement["devise"]:
        print(f"{str(investissement)}:  L'investissement a une valeur mais pas de devise.")


In [23]:
for key in result.keys():
    val = result[key]
    if val is list:
        if len(val) == 0:
            print(f"{key}: rien n'a été extrait pour ce champ - vérifier qualité d'extraction.")

### Vérifications sémantiques

In [24]:
import re
def number_in_text(number, text):
    num_str = str(number)
    
    if "." in num_str:
        integer_part, decimal_part = num_str.split(".")
        if decimal_part == '0':
            decimal_part = None
    elif "," in num_str:
        integer_part, decimal_part = num_str.split(",")
        if decimal_part == '0':
            decimal_part = None
    else:
        integer_part, decimal_part = num_str, None

    if decimal_part:
        pattern = rf"""
            (?<!\d)
            {integer_part}
            [\.,]
            {decimal_part}0*
            (?!\d)
        """
    else:
        pattern = rf"""
            (?<!\d)
            {integer_part}
            (?![\d\.,])
        """

    return re.search(pattern, text, re.VERBOSE) is not None


for reduction in result["objectifs_climatiques"]["reductions_emissions"]:
    if not number_in_text(reduction["valeur"], text):
        print(f"{reduction['entite']} {reduction['scope']}: Valeur {reduction['valeur']} extraite absente du texte source")
    if reduction["entite"] not in text:
        print(f"L'entité {reduction['entite']} apparait dans le résumé mais pas dans le texte source.")

### Validation croisée par LLM

In [25]:
validation_prompt_file = './prompts/validation.txt'
try:
    with open(validation_prompt_file, 'r', encoding='utf-8') as file:
        validation_prompt = file.read()
except FileNotFoundError:
    print(f"Erreur: Le fichier '{validation_prompt_file}' n'existe pas.")
except Exception as e:
    print(f"Une erreur s'est produite: {e}") 

validation_prompt = validation_prompt.replace("{TEXTE}", text)
validation_prompt = validation_prompt.replace("{JSON}", str(result))

In [26]:
validation_response = chat.send_message(validation_prompt)
validation_response.text

'Le JSON contient une incohérence systématique concernant les `source_excerpt`. Pour presque tous les éléments extraits, l\'extrait de texte fourni provient du "résumé structuré" intermédiaire plutôt que du document original fourni, contrairement à l\'instruction "Ajoute un court extrait exact du texte (max 20 mots) dans \'source_excerpt\' pour chaque élément extrait."\n\nPar exemple :\n- Dans `objectifs_climatiques.neutralite_carbone.source_excerpt`, l\'extrait est "décarboner leurs activités et les énergies distribuées, ... atteindre la Carboneutralité d\'ici 2050.", qui est une paraphrase du résumé et non une citation exacte du texte source.\n- Les `source_excerpt` pour `objectifs_climatiques.reductions_emissions` font référence aux "chiffres_cles_mentions" du résumé, au lieu des sections spécifiques du document original.\n- La même erreur se répète pour `indicateurs_emissions`, `investissements_transition`, `projets_renouvelables`, `risques_climatiques`, `strategies_adaptation`, `t

L'erreur trouvée par le LLM est correcte: il est vrai que les `source_excerpt` contiennent des extraits du résumé plutot que du texte original. Cela est du au processus en deux étapes que nous utilisons: d'abord résumer le document pour le raccourcir et ne retenir que les informations importantes, puis extraire les informations que nous recherchons. Un axe d'amélioration pourrait etre d'aller chercher les `source_excerpt` directement dans le texte original apres l'extraction: cela permettrait d'avoir des sources plus précises mais aussi de réaliser une vérification supplémentaire de l'exactitude des informations remontées.
Le LLM a également trouvé des descriptions de mesures qui sont trop détaillées pour ce projet. 